<a href="https://colab.research.google.com/github/Pimentaah/Projeto-Transporte/blob/desenvolvimento/notebook/note_principal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Processo de Imigração para o Google Colab**

In [74]:
import os
!pip install python-dotenv

os.chdir('/content/Projeto-Transporte')

**Consultando se estou na Branch certa para voltar a trabalhar**

In [75]:
!git branch --show-current

desenvolvimento


**Importando Dados GTFS para o Colab, fiz o pull para atualizar.**

In [76]:
%cd /content/Projeto-Transporte

!git pull origin desenvolvimento

/content
From https://github.com/Pimentaah/Projeto-Transporte
 * branch            desenvolvimento -> FETCH_HEAD
Already up to date.


In [77]:
import os

PASTA_GTFS = "data/_sptrans_gtfs"

print("Arquivos GTFS encontrados:")
print(os.listdir(PASTA_GTFS))

Arquivos GTFS encontrados:
['stops.txt', 'agency.txt', 'stop_times.txt', 'trips.txt', 'fare_attributes.txt', 'frequencies.txt', 'calendar.txt', 'fare_rules.txt', 'routes.txt', 'shapes.txt']


# 1. Configuração e conexão com a API Olho Vivo

Nesta etapa, realizo a configuração inicial do projeto, importei as bibliotecas que posteriomente irei precisar e estabeleço a conexão com a API Olho Vivo da SPTrans.

In [78]:
import os
import requests
import pandas as pd
from google.colab import userdata

TOKEN = userdata.get('TOKEN')

BASE_URL = "https://api.olhovivo.sptrans.com.br/v2.1"

print("Token encontrado:", TOKEN is not None)

Token encontrado: True


In [79]:
sessao = requests.Session()

resposta = sessao.post(
    f"{BASE_URL}/Login/Autenticar",
    params={"token": TOKEN},
    timeout=15
)

print("Código HTTP:", resposta.status_code)
print("Resposta:", resposta.text)

Código HTTP: 200
Resposta: true


In [80]:
resposta = sessao.get(f"{BASE_URL}/Linha/Buscar", params={"termosBusca": "8000"}, timeout=15) # Na variável resposta, vou pegar(Get) dentro da Base URl, o end-point
linhas = resposta.json()  # transforma o texto da resposta em lista do Python

tabela_linhas = pd.DataFrame(linhas) # Peguei a variavel LINHAS e transformei esse resultado em um DataFrame para se tornar uma tabela.
tabela_linhas

,cl,lc,lt,sl,tl,tp,ts
0,2506,False,8000,1,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA
1,35274,False,8000,2,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA
2,1273,False,8000,1,10,PÇA. RAMOS DE AZEVEDO,TERM. LAPA
3,34041,False,8000,2,10,PÇA. RAMOS DE AZEVEDO,TERM. LAPA


In [81]:
import os

print("Diretório atual:")
print(os.getcwd())

print("\nConteúdo da pasta atual:")
print(os.listdir())

print("\nConteúdo da raiz do projeto:")
print(os.listdir("/content/Projeto-Transporte"))

Diretório atual:
/content/Projeto-Transporte

Conteúdo da pasta atual:
['data', 'requirements.txt', '.env.example', '.git', '.gitignore', 'notebook', 'README.md']

Conteúdo da raiz do projeto:
['data', 'requirements.txt', '.env.example', '.git', '.gitignore', 'notebook', 'README.md']


In [82]:
codigo_linha = 35274

resposta = sessao.get(f"{BASE_URL}/Parada/BuscarParadasPorLinha", # Chamando(Get) End-point da Base_URL para retornar informações de Paradas das Linhas.
                      params={"codigoLinha": codigo_linha}, timeout=15) # Passei como parametro que a resposta deve receber informações guardadas no código da variável codigo_linha.
paradas = pd.DataFrame(resposta.json()) # Estou transformando a resposta que veio em HTTPS em uma lista Json (Dicionário) e trasformando o Json em uma "DF" de Tabela, guardando em uma variável.

paradas = paradas.rename(columns={   # Utilizando o ".rename", eu tratei nomes das colunas para ficar mais prático de análisar olhando os nomes.
    "cp": "codigo_parada",
    "np": "nome_parada",
    "ed": "endereco",
    "py": "latitude",
    "px": "longitude"
})

print("Quantidade de paradas:", len(paradas)) #Linha Básica para verificar o tamanho da tabela paradas utilzando "Len"
paradas.head(15)  # Por tendência o ".head" pega as 5 primeiras linhas da tabela, mas podemos passar no parametro quantas linhas podemos ver.

Quantidade de paradas: 13


,codigo_parada,nome_parada,endereco,latitude,longitude
0,614432,BABY BARIONY C/B,R PEDRO MACHADO,-23.527606,-46.671514
1,7014416,ANGELICA C/B,R DR. ALBUQUERQUE LINS,-23.534377,-46.654322
2,60014604,SESC POMPEIA C/B,VDTO POMPEIA/ R JOAQUIM FERREIRA,-23.525262,-46.682254
3,60016783,PARADA PALMEIRAS C/B,R PADRE ANTONIO TOMAS/ AV POMPEIA,-23.525653,-46.679493
4,60016785,ANTARTICA C/B,R ENGENHEIRO STEVENSON/ PC SOUSA ARANHA,-23.526220,-46.674187
5,60016789,PQ. DA ÁGUA BRANCA C/B,R DONA GERMAINE BUCHARD/ R MINISTRO GODOI,-23.529636,-46.667777
6,480014608,TIBERIO C/B,R TIBERIO/ R MENFIS,-23.522875,-46.688219
7,480014609,DUILIO C/B,R DUILIO/ R CLAUDIO,-23.522627,-46.692121
8,480014610,TENDAL DA LAPA C/B,R ESPARTACO/ R MORAIS E SILVA,-23.522339,-46.696246
9,700006085,ROSA E SILVA C/B,R OLIMPIA DE ALMEIDA PRADO/ R LOPES DE OLIVEIRA,-23.532629,-46.657465


# 2. Funções auxiliares para análise

    Nesta etapa, criei funções que serão usadas ao longo do projeto, principalmente para calcular distâncias geográficas e identificar paradas próximas de certos pontos.

In [83]:
from math import radians, sin, cos, asin, sqrt # Importei da biblioteca Math, algumas operações matemáticas que vou precisar.

def distancia_m(lat1, lon1, lat2, lon2):
    """Distância em metros entre dois pontos (fórmula de Haversine)."""
    Raio = 6371000  # O numero atribuído a variável representa o raio da Terra em metros
    dlat = radians(lat2 - lat1) # Guarda o resultado passado para Radianos da conta de (Lat2 - Lat1) que está cotada em Graus.
    dlon = radians(lon2 - lon1) # Guarda o resultado passado para Radianos da conta de (Lon2 - Lon1) que está cotada em Graus.


    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * Raio * asin(sqrt(a))

In [84]:

distancias = [] # Criando uma variável de lista para guardar as distancias
for i in range(1, len(paradas)): # Loop para pegar cada Parada da tabela Paradas
    anterior = paradas.iloc[i - 1] # Guardando o valor anterior do valor atual (i) do loop na variável "anterior".
    atual = paradas.iloc[i] # Guardando o valor atual do Loop (i) na variável "atual"
    distancias.append( # O append aqui serve para guardar na lista "distancia".
        distancia_m(anterior["latitude"], anterior["longitude"], # Dentro do append, chamei a função para calcular os parametros recebidos da latitude e longitude das variaveis "anterior" e "atual".
                    atual["latitude"], atual["longitude"])
    )

serie = pd.Series(distancias) # Guardei em uma variável e tranformei a lista "distancia" em uma Série com indices e valores para analisa melhor.
print(serie.describe()) # O describe serve para fazer uma análise Estatística sobre a Série "Distancias"
print("Saltos maiores que 1000 m:", (serie > 1000).sum(), "de", len(serie)) # Retornei utilizando um filtro, a contagem de distancias de são > 1000 e comparei com o tamanho da lista utilizando "Len".


count      12.000000
mean     1467.166558
std      1180.944655
min       284.829257
25%       513.911070
50%      1190.791606
75%      1984.562957
max      4116.000587
dtype: float64
Saltos maiores que 1000 m: 7 de 12


In [85]:
def paradas_proximas(tabela_paradas, lat, lon, raio_max): # Essa função serve para encontrar as paradas mais próximas do ponto de origem.
    """Devolve as paradas a até raio_max metros do ponto, da mais perto para a mais longe."""

    tabela = tabela_paradas.copy() # Copia o DataFrame original e salva em outra variável "teste", para não alterar o DataFrame Original.
    tabela["distancia_m"] = tabela.apply( # Estou criando uma coluna chamada "distancia_m" na nova tabela e estou aplicando uma Função com o apply.
        lambda parada: distancia_m(lat, lon, parada["latitude"], parada["longitude"]), axis=1 # Lamba é uma mini função dentro do apply que vai chamar a função distancia_m e passar os parametros das cordenadas do usuário e das paradas.
    ) # o "Axis = 1" significa que vai trabalhar o lambda pegando linha por linha, se fosse "0" seria para pegar de coluna em coluna

    return tabela[tabela["distancia_m"] <= raio_max].sort_values("distancia_m") # Aplicando um filtro para retornar uma tabela com apenas as distâncias que atendem o parametro de "Raio_max", e ordenando esses valores de forma ascentente.

# 3. Importação e preparação dos dados GTFS

    Nesta etapa, importo os arquivos GTFS da SPTrans e verifico a estrutura das tabelas, seus registros e a integridade das informações.
    
Percebi que somente com as buscas da API Olho Vivo, estavam retornando informações do tráfego de forma desorganizada, para isso, resolvi buscar na mesma documentação do site uma forma de retornar essas buscas mais estruturadas, e encontrei o GTFS, que funciona como um guia oficial de como as linhas são desenhadas, Ou seja, enquanto a API retornava somente poucas linhas, ordem "aleatória" e sem mapa de trajeto, O GTFS vai nos complementar essa informação de forma coletiva com  a API Olho vivo.



In [86]:
import os

PASTA_GTFS = "data/_sptrans_gtfs"

print(os.listdir(PASTA_GTFS))

['stops.txt', 'agency.txt', 'stop_times.txt', 'trips.txt', 'fare_attributes.txt', 'frequencies.txt', 'calendar.txt', 'fare_rules.txt', 'routes.txt', 'shapes.txt']


In [87]:
# Aqui eu estou carregando os arquivos GTFS

routes = pd.read_csv(f"{PASTA_GTFS}/routes.txt", dtype=str)
trips = pd.read_csv(f"{PASTA_GTFS}/trips.txt", dtype=str)
stops = pd.read_csv(f"{PASTA_GTFS}/stops.txt", dtype=str)
stop_times = pd.read_csv(f"{PASTA_GTFS}/stop_times.txt", dtype=str)

# Conferindo o conteúdo de cada tabela

for nome, tabela in [
    ("routes", routes),
    ("trips", trips),
    ("stops", stops),
    ("stop_times", stop_times)
]:
    print(f"\n{nome}")
    print("Quantidade de registros:", len(tabela))
    print("Colunas:", tabela.columns.tolist())



routes
Quantidade de registros: 1366
Colunas: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color']

trips
Quantidade de registros: 2268
Colunas: ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'shape_id']

stops
Quantidade de registros: 22263
Colunas: ['stop_id', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon']

stop_times
Quantidade de registros: 99076
Colunas: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence']


In [88]:

# Verificando a estrutura e alguns registros das tabelas

print("ROTAS")
display(routes.head(3))

print("VIAGENS")
display(trips.head(3))

print("PARADAS")
display(stops.head(3))

print("SEQUÊNCIA DAS PARADAS")
display(stop_times.head(3))


ROTAS


,route_id,agency_id,route_short_name,route_long_name,route_type,route_color,route_text_color
0,1012-10,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF
1,1012-21,1,1012-21,Term. Jd. Britânia - Jd. Rosinha,3,509E2F,FFFFFF
2,1014-10,1,1014-10,Cptm Perus - Morro Doce,3,509E2F,FFFFFF


VIAGENS


,route_id,service_id,trip_id,trip_headsign,direction_id,shape_id
0,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940
1,1012-21,U__,1012-21-0,Jd. Rosinha,0,85941
2,1014-10,U__,1014-10-0,Morro Doce,0,86069


PARADAS


,stop_id,stop_name,stop_desc,stop_lat,stop_lon
0,18848,Clínicas,NaN,-23.554007,-46.671097
1,18849,Vila Madalena,NaN,-23.546498,-46.691141
2,18850,Consolação,NaN,-23.558090,-46.660201


SEQUÊNCIA DAS PARADAS


,trip_id,arrival_time,departure_time,stop_id,stop_sequence
0,1012-10-0,07:00:00,07:00:00,301790,1
1,1012-10-0,07:01:20,07:01:20,301764,2
2,1012-10-0,07:02:40,07:02:40,301724,3


In [89]:

# Verificando os relacionamentos entre as tabelas GTFS:
# Estou verificando a inexistência dos requistos que pedi
# com o sinal de "~", esse sinal reprepresanta a inversão da verdade, ou seja, quero ver quantos numeros do que pedi
# estão sem um relacionamento, para isso utilizei a inversão de informação "~".

print("Viagens com route_id inexistente:")
print((~trips["route_id"].isin(routes["route_id"])).sum()) # Contando (.sum) se route_id está dentro de routes([route_id])  e invertendo com ~ para descobrir se existe algum relacionamento vazio

print("\nRegistros de stop_times com trip_id inexistente:")
print((~stop_times["trip_id"].isin(trips["trip_id"])).sum()) # Mesma relação com o comentário acima, porém com colunas diferentes.

print("\nRegistros de stop_times com stop_id inexistente:")
print((~stop_times["stop_id"].isin(stops["stop_id"])).sum()) # Mesma relação com o comentário acima, porém com colunas diferentes.


Viagens com route_id inexistente:
0

Registros de stop_times com trip_id inexistente:
0

Registros de stop_times com stop_id inexistente:
0


# 4. Relacionamento das tabelas GTFS

    Nesta etapa, utilizo relacionamentos entre as tabelas para reunir as informações de linhas, viagens e paradas em uma estrutura integrada.

Aqui eu inicio um processo interessante e importante para a conexão das informações (Tabelas), Nessa etapa do código, vou utilizar o conceito de Modelagem de Dados para relacionar as tabelas entre si.

In [90]:
#11

# Fazendo o  1° relacionamento de viagens com suas respectivas linhas

# Aplicando o conceito de modelagem de dados - Left Join

viagens_linhas = trips.merge( # Merge - Faz a mistura das duas tabelas
    routes,
    on="route_id", # Informa qual coluna irá conectar as tabelas
    how="left" # Conecta as informações mas mantém todos os registros da tabela "trips" a esquerda.
)

display(viagens_linhas.head()) # Apresentando como ficou a relação com as tabelas que foram conectadas.


,route_id,service_id,trip_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_color,route_text_color
0,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF
1,1012-21,U__,1012-21-0,Jd. Rosinha,0,85941,1,1012-21,Term. Jd. Britânia - Jd. Rosinha,3,509E2F,FFFFFF
2,1014-10,U__,1014-10-0,Morro Doce,0,86069,1,1014-10,Cptm Perus - Morro Doce,3,509E2F,FFFFFF
3,1015-10,USD,1015-10-0,Chác. Maria Trindade,0,81148,1,1015-10,Term. Jd. Britânia - Chác. Maria Trindade,3,509E2F,FFFFFF
4,1016-10,USD,1016-10-0,Shop. Center Norte,0,84176,1,1016-10,Cem. Do Horto - Shop. Center Norte,3,002F6C,FFFFFF


In [91]:
# 2° Relacionamento

# Adicionando as paradas de cada viagem

viagens_paradas = viagens_linhas.merge( # Relacionando as paradas de cada viagem
    stop_times,
    on="trip_id", # Conectando as informações pela referência ("trip_id")
    how="left" # Mantendo as informações da tabela "viagens_linhas" a Esquerda
)

display(viagens_paradas.head())


,route_id,service_id,trip_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_color,route_text_color,arrival_time,departure_time,stop_id,stop_sequence
0,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:00:00,07:00:00,301790,1
1,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:01:20,07:01:20,301764,2
2,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:02:40,07:02:40,301724,3
3,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:04:00,07:04:00,301730,4
4,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:05:20,07:05:20,30003042,5


In [92]:
# 3° Ultimo relacionamento - Estrutura Geral GTFS

# Adicionando nome e coordenadas dos pontos

gtfs_completo = viagens_paradas.merge( # Essa varaiável irá guardar todas as informações das relações entre tabelas que fizemos.
    stops, # Tabela paradas quue irá se relacionar com a tabela acima.
    on="stop_id", # As informações vão se conectar pela coluna de referencia "stop_id"
    how="left" # A tabela viagens_paradas irá manter as informações a esquerda, enquanto a tabela stops ficará a direita de acordo com a relação.
)

display(gtfs_completo.head())


,route_id,service_id,trip_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_color,route_text_color,arrival_time,departure_time,stop_id,stop_sequence,stop_name,stop_desc,stop_lat,stop_lon
0,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:00:00,07:00:00,301790,1,Terminal Jardim Britânia,NaN,-23.431924,-46.787131
1,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:01:20,07:01:20,301764,2,"R. Delsuc Alves De Magalhães, 194",Ref.: R. Diego Velasquez,-23.435427,-46.788460
2,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:02:40,07:02:40,301724,3,"R. Delsuc Alves De Magalhães, 538",Ref.: Estr. Cel. José Gladiador,-23.435469,-46.791522
3,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:04:00,07:04:00,301730,4,"Estr. Cel. José Gladiador, 262",Ref.: R. Delsuc Alves De Magalhães,-23.435386,-46.791885
4,1012-10,USD,1012-10-0,Jd. Monte Belo,0,85940,1,1012-10,Term. Jd. Britania - Jd. Monte Belo,3,509E2F,FFFFFF,07:05:20,07:05:20,30003042,5,"R. Virgínia Castiglioni, 381",Ref.: R Salvador Albano/ R Virginia Castiglioni,-23.433008,-46.792717


In [93]:

# Conferindo a tabela integrada

print("Quantidade de registros:", len(gtfs_completo))
print("\nColunas disponíveis:")
print(gtfs_completo.columns.tolist())

print("\nInformações da tabela:")
gtfs_completo.info()


Quantidade de registros: 99076

Colunas disponíveis:
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'shape_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon']

Informações da tabela:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99076 entries, 0 to 99075
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   route_id          99076 non-null  object
 1   service_id        99076 non-null  object
 2   trip_id           99076 non-null  object
 3   trip_headsign     99076 non-null  object
 4   direction_id      99076 non-null  object
 5   shape_id          99076 non-null  object
 6   agency_id         99076 non-null  object
 7   route_short_name  99076 non-null  object
 8   route_long_name   99076 non-null  object
 9   route_t

# 5. Análise exploratória das linhas e viagens

    Nesta etapa, realizo análises exploratórias dos dados GTFS para entender melhor a distribuição das linhas, a quantidade de viagens e a quantidade de paradas relacionadas a cada linha.

In [94]:
# Seleciona o primeiro trip_id disponível
viagem_exemplo = gtfs_completo["trip_id"].iloc[0]

print("Viagem selecionada:", viagem_exemplo)

# Atribuíndo valor ás variáveis

paradas_viagem = gtfs_completo[ # Essa variável só irá guardar as informações do DataFrame gtfs_compelto
    gtfs_completo["trip_id"] == viagem_exemplo # Guarda todos os id_linha do DataFrame que for == "viagem exemplo"
].copy() # Copiando o DataFrame "gtfs_completo" com a filtragem para não alterar o DataFrame original

paradas_viagem["stop_sequence"] = pd.to_numeric( # Alterando as sequncia de parada para tipo numero
    paradas_viagem["stop_sequence"]
)

paradas_viagem = paradas_viagem.sort_values( # Ordenando os valores (stop_sequence) e ordem ascendente.
    "stop_sequence"
)

display( # Aqui estou mostrando quais informações (Colunas) quero que retorne na minha busca.
    paradas_viagem[
        [
            "trip_id",
            "stop_sequence",
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon"
        ]
    ]
)

Viagem selecionada: 1012-10-0


,trip_id,stop_sequence,stop_id,stop_name,stop_lat,stop_lon
0,1012-10-0,1,301790,Terminal Jardim Britânia,-23.431924,-46.787131
1,1012-10-0,2,301764,"R. Delsuc Alves De Magalhães, 194",-23.435427,-46.788460
2,1012-10-0,3,301724,"R. Delsuc Alves De Magalhães, 538",-23.435469,-46.791522
3,1012-10-0,4,301730,"Estr. Cel. José Gladiador, 262",-23.435386,-46.791885
4,1012-10-0,5,30003042,"R. Virgínia Castiglioni, 381",-23.433008,-46.792717
5,1012-10-0,6,30003045,R. Nestor Gomes,-23.431541,-46.792973
6,1012-10-0,7,30003044,"R. Alberto Alvarez, 135",-23.430041,-46.794827
7,1012-10-0,8,30003043,"R. Dionísio Bellante, 29",-23.428500,-46.795379
8,1012-10-0,9,30003013,"R. João Fernandes Vieira, 92",-23.428531,-46.798702
9,1012-10-0,10,301693,"Estr. De Pirapora, 850",-23.431340,-46.800551


In [95]:
viagens_por_linha = ( # Guardando em uma variável a filtragem do algoritimo abaixo
    trips.groupby("route_id")["trip_id"] # Groupyby utilizado para agrupar as informações pelos identificadores passados. Trip_id é a coluna onde queremos analisar
    .nunique() # conta quantas viagem diferentes possui em cada linha
    .reset_index(name="quantidade_viagens") # Pega a filtragem e tranforma em uma tabela organizada, dando um nome para a coluna  a ser analisada.
)

display(viagens_por_linha.head(10))

,route_id,quantidade_viagens
0,1012-10,1
1,1012-21,1
2,1014-10,1
3,1015-10,1
4,1016-10,2
5,1017-10,2
6,1018-10,2
7,1019-10,2
8,1020-10,2
9,1021-10,2


In [96]:
paradas_por_linha = (
    gtfs_completo.groupby("route_id")["stop_id"] # Agrupando pelo route_id dando como referência de busca a coluna "stop_id"
    .nunique() # Contando quantas paradas diferentes possuem em cada linha agrupada
    .reset_index(name="quantidade_paradas") # Ordenando os valores em uma tabela organizada e atribuindo um nome para a coluna
)

display(paradas_por_linha.head(10))

,route_id,quantidade_paradas
0,1012-10,41
1,1012-21,17
2,1014-10,73
3,1015-10,20
4,1016-10,76
5,1017-10,116
6,1018-10,69
7,1019-10,70
8,1020-10,107
9,1021-10,58


In [97]:
resumo_linhas = ( # Fazendo um Merge de tabelas para resumir informações
    routes[["route_id", "route_short_name", "route_long_name"]] # Se referindo as colunas da tabela routes
    .merge(viagens_por_linha, on="route_id", how="left") # Relacionando os route_id que aparecem nas duas tabelas e relacionando essas informações mantendo a primeira tabela a esquerda.
    .merge(paradas_por_linha, on="route_id", how="left") # Faz um merge relacionado ao merge anterior e pega como referência o route_id para juntar as informações.
)

display(resumo_linhas.head(10))

,route_id,route_short_name,route_long_name,quantidade_viagens,quantidade_paradas
0,1012-10,1012-10,Term. Jd. Britania - Jd. Monte Belo,1.0,41.0
1,1012-21,1012-21,Term. Jd. Britânia - Jd. Rosinha,1.0,17.0
2,1014-10,1014-10,Cptm Perus - Morro Doce,1.0,73.0
3,1015-10,1015-10,Term. Jd. Britânia - Chác. Maria Trindade,1.0,20.0
4,1016-10,1016-10,Cem. Do Horto - Shop. Center Norte,2.0,76.0
5,1017-10,1017-10,Perus - Conexão Vl. Iório,2.0,116.0
6,1018-10,1018-10,Vl. Rosa - Metrô Santana,2.0,69.0
7,1019-10,1019-10,Sol Nascente - Term. Pirituba,2.0,70.0
8,1020-10,1020-10,Perus - Conexão Vl. Iório,2.0,107.0
9,1021-10,1021-10,Cohab Brasilândia - Term. Pirituba,2.0,58.0


In [98]:
ranking_linhas = resumo_linhas.sort_values( # Variavel que irá rankear o algoritimo abaixo
    by="quantidade_viagens", # Selecionando a coluna ("quantidade_viagens") da variável resumo_linhas
    ascending=False # Ordenando de forma descentente as informações.
)

display(ranking_linhas.head(10)) # Retornando as 10 primeiras linhas da variável.

,route_id,route_short_name,route_long_name,quantidade_viagens,quantidade_paradas
1352,N802-11,N802-11,Term. Pinheiros - Term. Pq. D. Pedro Ii,2.0,55.0
1335,N731-11,N731-11,Term. Capelinha - Term. Sto. Amaro,2.0,35.0
1334,N706-11,N706-11,Term. Campo Limpo - Term. Pinheiros,2.0,64.0
1333,N705-11,N705-11,Term. Sto. Amaro - Term. Pinheiros,2.0,44.0
1332,N704-11,N704-11,Term. Capelinha - Term. Sto. Amaro,2.0,32.0
1331,N703-11,N703-11,Term. Jd. âNgela - Term. Sto. Amaro,2.0,38.0
1330,N702-11,N702-11,Term. Sto. Amaro - Term. Pq. D. Pedro Ii,2.0,57.0
1329,N701-11,N701-11,Term. Sto. Amaro - Term. Pq. D. Pedro Ii,2.0,64.0
31,119L-10,119L-10,Vl. Sulina - Term. Lapa,2.0,59.0
30,119C-10,119C-10,Pq. Edu Chaves - Term. Princ. Isabel,2.0,113.0


In [99]:
linhas_sem_viagens = resumo_linhas[
    resumo_linhas["quantidade_viagens"].isna() # Buscando somente os registros na coluna quantidade_viagens que possuem valor nulo
]

display(linhas_sem_viagens) # Mostrando as linhas guardadas como nulas da busca acima

,route_id,route_short_name,route_long_name,quantidade_viagens,quantidade_paradas
88,177H-21,177H-21,Metrô Santana - Pinheiros,NaN,NaN
107,189L-21,189L-21,Conexão Vl. Iório - Lapa,NaN,NaN
154,2100-21,2100-21,Term. Vl. Carrão - Term. Pq. D. Pedro Ii,NaN,NaN
158,2105-21,2105-21,Jd. Filhos Da Terra - Metrô Santana,NaN,NaN
170,220P-10,220P-10,Term. Parada Inglesa - Ctn,NaN,NaN
336,3686-21,3686-21,Jd. São Paulo - Metrô Vl. Matilde,NaN,NaN
438,407A-21,407A-21,Pq. Savoy - Metrô Belém,NaN,NaN
444,407I-21,407I-21,Conj. Manoel Da Nóbrega - Metrô Tatuapé,NaN,NaN
478,4312-21,4312-21,Jd. Marília - Metrô Belém,NaN,NaN
481,4314-21,4314-21,Inácio Monteiro - Term. Vl. Carrão,NaN,NaN


# 6. Identificação de paradas próximas à origem e ao destino

    Nesta etapa, utilizo coordenadas de origem e destino para identificar as paradas de transporte público próximas a esses pontos, considerando um raio de distância definido.

In [100]:
# Buscando colunas de uma variável e guardando em outra variável para realizar uma analise sem alterar a variável com infos importantes.
paradas_gtfs = stops[
    ["stop_id", "stop_name", "stop_lat", "stop_lon"]
].copy()

#Alterando o nome das colunas para um nome intuítivo
paradas_gtfs = paradas_gtfs.rename(columns={
    "stop_lat": "latitude",
    "stop_lon": "longitude"
})

# Tranformando o tipo de dado das colunas e convertendo para Número
paradas_gtfs["latitude"] = pd.to_numeric(paradas_gtfs["latitude"])
paradas_gtfs["longitude"] = pd.to_numeric(paradas_gtfs["longitude"])

## 6.1 Definição das coordenadas de origem e destino

Neste bloco de código, defini as coordenadas geográficas de origem e destino e coloquei o raio de busca usado para localizar as paradas próximas a esses pontos.


In [101]:
# Coordenadas de exemplo para o primeiro teste do algoritmo
origem = {
    "latitude": -23.5505,
    "longitude": -46.6333
}

destino = {
    "latitude": -23.5614,
    "longitude": -46.6559
}

raio_busca = 500

In [102]:
# Chamando função que está recebendo parametros de Origem e guardando o resultado em uma variável
paradas_origem = paradas_proximas(
    paradas_gtfs,
    origem["latitude"],
    origem["longitude"],
    raio_busca
)

# Chamando função que está recebendo parametros de destino e guardando os resultados em uma variável
paradas_destino = paradas_proximas(
    paradas_gtfs,
    destino["latitude"],
    destino["longitude"],
    raio_busca
)

# Chamando as variáveis que guardaram os resultados da função
print("Paradas próximas da origem:", len(paradas_origem))
print("Paradas próximas do destino:", len(paradas_destino))

Paradas próximas da origem: 67
Paradas próximas do destino: 16


## 6.2 Identificação das linhas próximas à origem e ao destino
    
A partir das paradas encontradas, encontrei quais linhas atendem às regiões de origem e destino. Depois, verifico quais linhas aparecem nos dois conjuntos.

In [103]:
# Bloco utilizado para identificar e guardar quais linhas passam próximas á origem que o passageiro está.
linhas_origem = set( # O set é usado para tranformar informações e um conjunto sem duplicadas, ou seja, não repetir as linhas retornadas.
    gtfs_completo[
        gtfs_completo["stop_id"].isin(paradas_origem["stop_id"]) # Verificando quais valores da tabela gtfs_completo está na tabela parada origem.
    ]["route_id"].unique() # Retornando valores unicos da coluna route_id.
)

# Bloco utilizado para verificar quais linhas passam próximas ao destino, sem haver linhas duplicadas.
linhas_destino = set(
    gtfs_completo[
        gtfs_completo["stop_id"].isin(paradas_destino["stop_id"])
    ]["route_id"].unique()
)

linhas_candidatas = linhas_origem.intersection(linhas_destino) # Verificando e guardando a intersseção de identificadores nas variáveis.

print("Linhas próximas da origem:", len(linhas_origem))
print("Linhas próximas do destino:", len(linhas_destino))
print("Linhas em comum:", len(linhas_candidatas))

Linhas próximas da origem: 99
Linhas próximas do destino: 45
Linhas em comum: 5


## 7. Identificação de viagens diretas
Nesta etapa, relaciono as paradas próximas à origem e ao destino para identificar viagens que permitem fazer o percurso sem necessidade de troca de ônibus.

In [104]:
# Identificando as paradas de embarque
embarques = gtfs_completo[
    gtfs_completo["stop_id"].isin(paradas_origem["stop_id"]) # Verificando quais valores da tabela gtfs_completo está na tabela parada origem.
][["route_id", "trip_id", "stop_id", "stop_sequence"]].copy() # Aqui eu estou filtrando quais são as colunas que me interessam e estou guardando em uma variável para não alterar o DF original.

# Identificando as paradas de desembarque
desembarques = gtfs_completo[
    gtfs_completo["stop_id"].isin(paradas_destino["stop_id"])
][["route_id", "trip_id", "stop_id", "stop_sequence"]].copy()

# Nesse bloco eu estou convertendo as colunas para número, futuramente será utilizada para comparações numéricas.
embarques["stop_sequence"] = pd.to_numeric(embarques["stop_sequence"])
desembarques["stop_sequence"] = pd.to_numeric(desembarques["stop_sequence"])

# Relacionando embarques e desembarques da mesma viagem
viagens_diretas = embarques.merge( # Estou relacioando as informações que se referem a mesma viagem.
    desembarques,
    on=["route_id", "trip_id"], # Nessa linha, mantive as informações que se referem a mesma viagem, no caso, os mesmo identidicadores de linha e viagem.
    suffixes=("_origem", "_destino") # Separei as informações de embarque e desembarque com sufixos para não confundir as informações que se referem a mesma viagem.
)

# Mantendo apenas viagens em que o destino vem depois da origem, ou seja, quero manter apenas viagens que o stop_sequence destino é maior que o de origem.
viagens_diretas = viagens_diretas[
    viagens_diretas["stop_sequence_destino"] > # Fazendo a comparação entre as colunas stop_sequence_destino e stop_sequence_origem, para manter apenas viagens que o destino vem depois da origem.
    viagens_diretas["stop_sequence_origem"]
]

display(viagens_diretas.head(10))

,route_id,trip_id,stop_id_origem,stop_sequence_origem,stop_id_destino,stop_sequence_destino
8,6403-10,6403-10-1,800016488,1,70016563,8
13,805L-10,805L-10-0,800016587,10,70016916,29
14,805L-10,805L-10-0,800016587,10,70015949,30
15,805L-10,805L-10-0,800016587,10,260016919,31
21,N701-11,N701-11-1,8010181,1,70016563,14


## 7.1 Complementação das informações de embarque e desembarque

Acrescento às opções de viagem os nomes das paradas de embarque e desembarque, suas respectivas distâncias e as informações das linhas.

In [105]:
## Identificação de parada de embarque

# Nesse bloco estou fazendo um merge entre a tabela de viagens_diretas e a tabela paradas_origem, para trazer informações de nome da parada e distância da parada de embarque.
opcoes_viagem = viagens_diretas.merge(
    paradas_origem[["stop_id", "stop_name", "distancia_m"]],
    left_on="stop_id_origem", # o left on serve para relacionar qual coluna da tabela da esquerda vai se relacionar com a tabela da direita.
    right_on="stop_id", # O right on serve para relacionar qual informação da coluna da direita vai se relacionar com a coluna da esquerda.
    how="left" # Vou manter as informações da tabela da esquerda.
)

# Renomeando colunas para nomes mais entendiveis e intuitivos.
opcoes_viagem = opcoes_viagem.rename(columns={
    "stop_name": "parada_embarque",
    "distancia_m": "distancia_origem"
})

opcoes_viagem = opcoes_viagem.drop(columns="stop_id") # Apagando a coluna stop_id que não é mais útil.

display(opcoes_viagem.head())

,route_id,trip_id,stop_id_origem,stop_sequence_origem,stop_id_destino,stop_sequence_destino,parada_embarque,distancia_origem
0,6403-10,6403-10-1,800016488,1,70016563,8,Terminal Parque Dom Pedro II - Plat 01,452.009942
1,805L-10,805L-10-0,800016587,10,70016916,29,Fórum João Mendes,215.794707
2,805L-10,805L-10-0,800016587,10,70015949,30,Fórum João Mendes,215.794707
3,805L-10,805L-10-0,800016587,10,260016919,31,Fórum João Mendes,215.794707
4,N701-11,N701-11-1,8010181,1,70016563,14,"Terminal Parque Dom Pedro II - Plat 01, 0",450.252803


In [106]:
## Identificação de parada de desembarque

opcoes_viagem = opcoes_viagem.merge(
    paradas_destino[["stop_id", "stop_name", "distancia_m"]], # Filtrando as colunas que quero da tabela de destino.
    left_on="stop_id_destino", # Pegando a coluna que quero usar de referencia da tabela da esquerda.
    right_on="stop_id", # Pegando a coluna que quero usar de referencia da tabela da direita.
    how="left" # Vou manter as informações da tabela da esquerda.
)

opcoes_viagem = opcoes_viagem.rename(columns={ # Renomeando colunas para nomes mais entendiveis e intuitivos.
    "stop_name": "parada_desembarque",
    "distancia_m": "distancia_destino"
})

opcoes_viagem = opcoes_viagem.drop(columns="stop_id") # Excluindo a coluna stop_id, pois usamos ela acima somente para fazer a comparação.

display(opcoes_viagem.head())

,route_id,trip_id,stop_id_origem,stop_sequence_origem,stop_id_destino,stop_sequence_destino,parada_embarque,distancia_origem,parada_desembarque,distancia_destino
0,6403-10,6403-10-1,800016488,1,70016563,8,Terminal Parque Dom Pedro II - Plat 01,452.009942,Parada 1 - Getúlio Vargas C/B,212.583967
1,805L-10,805L-10-0,800016587,10,70016916,29,Fórum João Mendes,215.794707,"Av. Paulista, 1374",226.916766
2,805L-10,805L-10-0,800016587,10,70015949,30,Fórum João Mendes,215.794707,"Av. Paulista, 1754",192.810648
3,805L-10,805L-10-0,800016587,10,260016919,31,Fórum João Mendes,215.794707,Parada 2 - Frei Caneca,443.937049
4,N701-11,N701-11-1,8010181,1,70016563,14,"Terminal Parque Dom Pedro II - Plat 01, 0",450.252803,Parada 1 - Getúlio Vargas C/B,212.583967


In [107]:
opcoes_viagem = opcoes_viagem.merge( # Pegando informações da tabela de rotas e relacionando com a tabela opções de viagem.
    routes[["route_id", "route_short_name", "route_long_name"]], # Selecionando quais são as colunas que quero olhar da tabela rotas.
    on="route_id", # Usando como referencia a coluna route_id para conseguir relacionar as informações das tabelas.
    how="left" # Mantendo as informações da tabela da esquerda.(opções_viagem)
)

opcoes_viagem = opcoes_viagem.rename(columns={ # Renomeando colunas.
    "route_short_name": "numero_linha",
    "route_long_name": "nome_linha"
})

display(opcoes_viagem.head())

,route_id,trip_id,stop_id_origem,stop_sequence_origem,stop_id_destino,stop_sequence_destino,parada_embarque,distancia_origem,parada_desembarque,distancia_destino,numero_linha,nome_linha
0,6403-10,6403-10-1,800016488,1,70016563,8,Terminal Parque Dom Pedro II - Plat 01,452.009942,Parada 1 - Getúlio Vargas C/B,212.583967,6403-10,Term. João Dias - Term. Pq. D. Pedro Ii
1,805L-10,805L-10-0,800016587,10,70016916,29,Fórum João Mendes,215.794707,"Av. Paulista, 1374",226.916766,805L-10,Term. Princ. Isabel - Aclimação
2,805L-10,805L-10-0,800016587,10,70015949,30,Fórum João Mendes,215.794707,"Av. Paulista, 1754",192.810648,805L-10,Term. Princ. Isabel - Aclimação
3,805L-10,805L-10-0,800016587,10,260016919,31,Fórum João Mendes,215.794707,Parada 2 - Frei Caneca,443.937049,805L-10,Term. Princ. Isabel - Aclimação
4,N701-11,N701-11-1,8010181,1,70016563,14,"Terminal Parque Dom Pedro II - Plat 01, 0",450.252803,Parada 1 - Getúlio Vargas C/B,212.583967,N701-11,Term. Sto. Amaro - Term. Pq. D. Pedro Ii


In [108]:
opcoes_viagem["distancia_total"] = ( # Criando uma nova coluna que irá guardar a soma das distâncias de origem e destino.
    opcoes_viagem["distancia_origem"] + opcoes_viagem["distancia_destino"] # Somando as informalções das colunas.
)

opcoes_viagem = opcoes_viagem.sort_values( # Ordenando de forma crescente a tabela, tendo como referencia a coluna distancia_total. Priorizando as viagens que possuem menor distância total.
    by="distancia_total"
)

display(opcoes_viagem.head(10))

,route_id,trip_id,stop_id_origem,stop_sequence_origem,stop_id_destino,stop_sequence_destino,parada_embarque,distancia_origem,parada_desembarque,distancia_destino,numero_linha,nome_linha,distancia_total
2,805L-10,805L-10-0,800016587,10,70015949,30,Fórum João Mendes,215.794707,"Av. Paulista, 1754",192.810648,805L-10,Term. Princ. Isabel - Aclimação,408.605355
1,805L-10,805L-10-0,800016587,10,70016916,29,Fórum João Mendes,215.794707,"Av. Paulista, 1374",226.916766,805L-10,Term. Princ. Isabel - Aclimação,442.711473
3,805L-10,805L-10-0,800016587,10,260016919,31,Fórum João Mendes,215.794707,Parada 2 - Frei Caneca,443.937049,805L-10,Term. Princ. Isabel - Aclimação,659.731756
4,N701-11,N701-11-1,8010181,1,70016563,14,"Terminal Parque Dom Pedro II - Plat 01, 0",450.252803,Parada 1 - Getúlio Vargas C/B,212.583967,N701-11,Term. Sto. Amaro - Term. Pq. D. Pedro Ii,662.836769
0,6403-10,6403-10-1,800016488,1,70016563,8,Terminal Parque Dom Pedro II - Plat 01,452.009942,Parada 1 - Getúlio Vargas C/B,212.583967,6403-10,Term. João Dias - Term. Pq. D. Pedro Ii,664.593908


## 8. Seleção da viagem e construção do itinerário
    
    Nesta etapa, organizo as opções de viagem pela distância total de caminhada, seleciono a melhor opção encontrada e recupero as informações necessárias para apresentar o percurso ao usuário.

Tomei como foco focar no retorno da melhor viagem/linha para o usuário, mostrar trajetos que o onibus irá fazer e retornar informações importantes para o melhor caminha retornado.

In [109]:
opcao_selecionada = opcoes_viagem.iloc[0] # Retornando a primeira linha da tabela opcoes_viagem, que é a viagem com menor distância total.

display(opcao_selecionada) # retornando informações da viagem.

,2
route_id,805L-10
trip_id,805L-10-0
stop_id_origem,800016587
stop_sequence_origem,10
stop_id_destino,70015949
stop_sequence_destino,30
parada_embarque,Fórum João Mendes
distancia_origem,215.794707
parada_desembarque,"Av. Paulista, 1754"
distancia_destino,192.810648


## 8.1 Recuperação do itinerário

Recupero as paradas pertencentes à viagem selecionada e filtro só aquelas que estão entre o ponto de embarque e o ponto de desembarque.

In [110]:
paradas_itinerario = gtfs_completo[ # Copiando as informações da tabela gtfs que são iguais ao trip_id da viagem selecionada.
    gtfs_completo["trip_id"] == opcao_selecionada["trip_id"]
].copy()

paradas_itinerario["stop_sequence"] = pd.to_numeric( # Nesse bloco estou convertendo a coluna para numero.
    paradas_itinerario["stop_sequence"]
)

paradas_itinerario = paradas_itinerario[ # Filtrando as informações da tabela para trazer apenas as paradas que estão entre a origem e o destino da viagem selecionada.
    (paradas_itinerario["stop_sequence"] >= opcao_selecionada["stop_sequence_origem"]) &
    (paradas_itinerario["stop_sequence"] <= opcao_selecionada["stop_sequence_destino"])
]

paradas_itinerario = paradas_itinerario.sort_values( # Ordenando as informações da tabela de forma ascendente, sendo "stop_sequence" a coluna de referência.
    by="stop_sequence"
)

display( # Estou retornando apenas as colunas que quero analisar.
    paradas_itinerario[
        ["stop_sequence", "stop_id", "stop_name", "stop_lat", "stop_lon"]
    ]
)

,stop_sequence,stop_id,stop_name,stop_lat,stop_lon
75726,10,800016587,Fórum João Mendes,-23.552429,-46.633532
75727,11,800008375,"R. Cons. Furtado, 256",-23.555143,-46.633160
75728,12,490008377,"R. Cons. Furtado, 534",-23.557722,-46.633042
75729,13,490008380,"R. Cons. Furtado, 718",-23.559348,-46.632902
75730,14,490001985,"R. Cons. Furtado, 1004",-23.561983,-46.632663
75731,15,490017008,"R. Cons. Furtado, 1246",-23.564112,-46.632510
75732,16,490013041,"Av. Da Aclimação, 120",-23.568159,-46.633387
75733,17,4910065,"Av. Da Aclimação, 344",-23.569531,-46.632681
75734,18,4910016,"Av. Da Aclimação, 666",-23.572502,-46.633076
75735,19,490001793,"Av. Da Aclimação, 876",-23.573464,-46.631577


## 8.2 Identificação do sentido da viagem

Consulto as informações da viagem selecionada para identificar seu sentido e o destino indicado no itinerário.

In [111]:
dados_viagem = trips[ # Estou selecionando as informações da tabela trips que são iguais as informações da opcao_selecionada.
    trips["trip_id"] == opcao_selecionada["trip_id"]
][["trip_id", "direction_id", "trip_headsign"]].drop_duplicates() #Selecionando as colunas que quero guardar e Removendo informações duplicadas.

display(dados_viagem)

,trip_id,direction_id,trip_headsign
1739,805L-10-0,0,Aclimação


## 9. Integração com a API Olho Vivo
    Nesta etapa, uso as informações obtidas no GTFS para realizar consultas à API Olho Vivo e verificar como os dados da viagem podem ser relacionados aos registros que foram dados pela API.



## 9.1 Validação da parada de embarque na API

In [112]:
# Recuperando as coordenadas da parada de embarque no GTFS
# 1 Validação da parada na API.


parada_embarque_gtfs = stops[
    stops["stop_id"] == opcao_selecionada["stop_id_origem"]
].iloc[0]

# Buscando essa parada na API Olho Vivo pelo nome
resposta_parada_api = sessao.get(
    f"{BASE_URL}/Parada/Buscar",
    params={"termosBusca": opcao_selecionada["parada_embarque"]},
    timeout=15
)

resposta_parada_api.raise_for_status()

paradas_api_busca = pd.DataFrame(resposta_parada_api.json())

# Padronizando os nomes das colunas da API
paradas_api_busca = paradas_api_busca.rename(columns={
    "cp": "codigo_parada",
    "np": "nome_parada",
    "ed": "endereco",
    "py": "latitude",
    "px": "longitude"
})

if paradas_api_busca.empty:
    print("A API não encontrou paradas com esse nome.")
else:
    # Calculando a distância entre a parada GTFS e cada resultado da API
    paradas_api_busca["distancia_m"] = paradas_api_busca.apply(
        lambda parada: distancia_m(
            float(parada_embarque_gtfs["stop_lat"]),
            float(parada_embarque_gtfs["stop_lon"]),
            parada["latitude"],
            parada["longitude"]
        ),
        axis=1
    )

    paradas_api_busca = paradas_api_busca.sort_values(
        by="distancia_m"
    )

    display(
        paradas_api_busca[
            ["codigo_parada", "nome_parada", "endereco", "distancia_m"]
        ]
    )

A API não encontrou paradas com esse nome.


## 9.2 Identificação da linha na API

In [113]:
# 2 Identificação da linha na API.

resposta_linha_api = sessao.get(
    f"{BASE_URL}/Linha/Buscar",
    params={"termosBusca": str(opcao_selecionada["numero_linha"])},
    timeout=15
)

resposta_linha_api.raise_for_status()

linhas_api = pd.DataFrame(resposta_linha_api.json())

display(linhas_api)

,cl,lc,lt,sl,tl,tp,ts
0,124,True,805L,1,10,ACLIMAÇÃO,TERM. PRINC. ISABEL
1,32892,True,805L,2,10,ACLIMAÇÃO,TERM. PRINC. ISABEL


## 9.3 Consulta de previsões da linha na API

In [114]:
# 7 Comparação das coordenadas das paradas.

codigo_linha_api = 32892  # linha sentido Aclimação

resposta = sessao.get(
    f"{BASE_URL}/Previsao/Linha",
    params={"codigoLinha": codigo_linha_api},
    timeout=15
)
resposta.raise_for_status()

pontos_previsao = pd.DataFrame(resposta.json()["ps"])

pontos_previsao["distancia_gtfs"] = pontos_previsao.apply(
    lambda parada: distancia_m(
        float(parada_embarque_gtfs["stop_lat"]),
        float(parada_embarque_gtfs["stop_lon"]),
        float(parada["py"]),
        float(parada["px"])
    ),
    axis=1
)

pontos_previsao = pontos_previsao.sort_values(
    by="distancia_gtfs"
)

display(
    pontos_previsao[
        ["cp", "np", "py", "px", "distancia_gtfs", "vs"]
    ]
)

,cp,np,py,px,distancia_gtfs,vs
1,70016834,,-23.568023,-46.648338,2298.714073,[]
2,706304,,-23.565900,-46.650910,2319.732454,[]
0,70016782,,-23.570405,-46.645227,2327.284167,[]
3,70016916,,-23.562931,-46.654428,2429.008489,[]
4,70015949,,-23.560303,-46.657365,2582.233400,[]
7,260016919,PARADA 2 - FREI CANECA,-23.558768,-46.659175,2707.148114,[]
5,260016920,,-23.556066,-46.662369,2967.057568,[]
6,2600690,,-23.555192,-46.663264,3046.137982,[]


## 10. Análises exploratórias e testes da API

        Organizei as células que usei para investigar os recursos da API e entender os dados que ela disponibiliza.

In [115]:
# 3 Investigar as paradas retornadas para cada linha.

codigos_linha = linhas_api["cl"].tolist()

for codigo in codigos_linha:
    resposta = sessao.get(
        f"{BASE_URL}/Parada/BuscarParadasPorLinha",
        params={"codigoLinha": int(codigo)},
        timeout=15
    )

    resposta.raise_for_status()

    paradas_linha_api = pd.DataFrame(resposta.json())

    print(f"\nCódigo da linha: {codigo}")
    display(paradas_linha_api.head())


Código da linha: 124


,cp,np,ed,py,px
0,670013739,DUQUE DE CAXIAS - B/C,R GENERAL OSORIO/ AV DUQUE DE CAXIAS,-23.537246,-46.641958



Código da linha: 32892


""


In [116]:
# 4 Verificar as previsões disponíveis para várias linhas.
for codigo in linhas_api["cl"]:
    resposta = sessao.get(
        f"{BASE_URL}/Previsao/Linha",
        params={"codigoLinha": int(codigo)},
        timeout=15
    )

    resposta.raise_for_status()

    dados_previsao = resposta.json()

    print(f"\nCódigo da linha: {codigo}")
    print("Horário de referência:", dados_previsao.get("hr"))
    print("Pontos retornados:", len(dados_previsao.get("ps", [])))


Código da linha: 124
Horário de referência: 15:19
Pontos retornados: 1

Código da linha: 32892
Horário de referência: 15:19
Pontos retornados: 8


In [117]:
# 5 Testar a consulta de previsão de uma linha específica (32892).
resposta = sessao.get(
    f"{BASE_URL}/Previsao/Linha",
    params={"codigoLinha": 32892},
    timeout=15
)

resposta.raise_for_status()

dados_previsao = resposta.json()

pontos_previsao = pd.DataFrame(dados_previsao["ps"])

display(pontos_previsao)

,cp,np,py,px,vs
0,70016782,,-23.570405,-46.645227,[]
1,70016834,,-23.568023,-46.648338,[]
2,706304,,-23.565900,-46.650910,[]
3,70016916,,-23.562931,-46.654428,[]
4,70015949,,-23.560303,-46.657365,[]
5,260016920,,-23.556066,-46.662369,[]
6,2600690,,-23.555192,-46.663264,[]
7,260016919,PARADA 2 - FREI CANECA,-23.558768,-46.659175,[]


In [118]:
# 9 Consulta de previsão da parada, após a reformulação.
resposta_parada = sessao.get(
    f"{BASE_URL}/Previsao/Parada",
    params={"codigoParada": 70015949},
    timeout=15
)

resposta_parada.raise_for_status()

dados_previsao_parada = resposta_parada.json()

print(dados_previsao_parada)

{'hr': '15:19', 'p': {'cp': 70015949, 'np': '', 'py': -23.560303, 'px': -46.657365, 'l': [{'c': '875A-10', 'cl': 609, 'sl': 1, 'lt0': 'PERDIZES', 'lt1': 'AEROPORTO', 'qv': 2, 'vs': [{'p': '64383', 't': '15:23', 'a': True, 'ta': '2026-11-24T18:19:02Z', 'py': -23.56357666666667, 'px': -46.653594999999996, 'sv': None, 'is': None}, {'p': '64480', 't': '16:21', 'a': True, 'ta': '2026-11-24T18:18:53Z', 'py': -23.622006666666667, 'px': -46.661635, 'sv': None, 'is': None}]}, {'c': '917H-10', 'cl': 33224, 'sl': 2, 'lt0': 'METRÔ VL. MARIANA', 'lt1': 'TERM. PIRITUBA', 'qv': 6, 'vs': [{'p': '11477', 't': '15:24', 'a': True, 'ta': '2026-11-24T18:18:43Z', 'py': -23.564664999999998, 'px': -46.65245333333334, 'sv': None, 'is': None}, {'p': '11905', 't': '15:25', 'a': True, 'ta': '2026-11-24T18:19:14Z', 'py': -23.565243333333335, 'px': -46.65125333333334, 'sv': None, 'is': None}, {'p': '11907', 't': '15:29', 'a': True, 'ta': '2026-11-24T18:18:53Z', 'py': -23.566466666666663, 'px': -46.65041166666666, '